In [1]:
import biosteam as bst
import thermosteam as tmo
import numpy as np
from lignin_saf.ligsaf_units import SolvolysisReactor, HydrogenolysisReactor, PSA, CatalystMixer
from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import (
    rcf_oil_yield, prices, feed_parameters, rcf_conditions,
    solvolysis_parameters, meoh_h2o, h2_biomass_ratio, RCF_catalyst,
    poplar_density, free_frac,
    V_max_limit, hexane_partition_IDs, hexane_partition_K, hexane_price, hexane_purification
)
from lignin_saf.ligsaf_utilities_system import create_rcf_utilities_system

from biosteam import main_flowsheet as F
bst.nbtutorial()

In [2]:
# Code just to increase the number of display units for the various components
tmo.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
tmo.MultiStream.display_units.N = 40  
bst.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
bst.MultiStream.display_units.N = 40  

In [3]:
#def create_rcf_system(ins=None):
"""
Build and return the RCF loop as a bst.System.

Parameters
----------
ins : bst.Stream, optional
    Poplar feedstock stream. If None, a default stream is created
    from feed_parameters in ligsaf_settings.py.

Returns
-------
bst.System
    'RCF_System' with meoh_recycle and hydrogen_recycle converged.

Notes
-----
Caller is responsible for setting up thermodynamics before calling:
    chems = create_chemicals()
    bst.settings.set_thermo(chems)
    bst.settings.CEPCI = 541.7
"""

chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7

chems = bst.settings.chemicals

# Poplar pseudocomponent group (Bartling et al Table S1)
# Re-defining is safe — thermosteam silently overwrites existing groups
chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
            'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                    0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

# ── Feedstock ─────────────────────────────────────────────────────────────

ins = bst.Stream('Poplar_In',
                    Poplar=feed_parameters['flow'] * 1e3,
                    Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                    phase='l', units='kg/d')

# ── Recycle streams ───────────────────────────────────────────────────────
meoh_recycle = bst.MultiStream('Meoh_recycle', phases=('s', 'l', 'g'))
hydrogen_recycle = bst.Stream('hydrogen_recycle', P=3e6, phase='g')

# ── Co-feeds ──────────────────────────────────────────────────────────────
meoh_in = bst.Stream('Meoh_in', Methanol=0.0, Water=0.0, phase='l', units='L/d')

hydrogen_in = bst.Stream('Hydrogen_In',
                            Hydrogen=h2_biomass_ratio * 2e6,
                            units='kg/day',
                            T=80 + 273.15,   # 80°C PEM electrolyzer outlet
                            P=3e6,           # 30 bar PEM electrolyzer outlet
                            phase='g')

# ── Unit operations ───────────────────────────────────────────────────────

# MeOH mixer: adjusts fresh feed to make up for what the recycle doesn't supply
meoh_h2o_mix = bst.units.Mixer('MIX100', ins=(meoh_in, meoh_recycle), rigorous=True)

@meoh_h2o_mix.add_specification(run=True)
def meoh_water_flow():
    fresh_solvent = meoh_h2o_mix.ins[0]
    recycle_solvent = meoh_h2o_mix.ins[1]
    total_vol_hr = solvolysis_reactor.compute_Q_total()  # m³/hr — derived from bed geometry
    meoh_flow_mol = (
        total_vol_hr * meoh_h2o / (meoh_h2o + 1)
        * chems['Methanol'].rho(phase='l', T=rcf_conditions['T'], P=rcf_conditions['P'])
        * (1 / chems['Methanol'].MW)
    )
    water_flow_mol = (
        total_vol_hr / (meoh_h2o + 1)
        * chems['Water'].rho(phase='l', T=rcf_conditions['T'], P=rcf_conditions['P'])
        * (1 / chems['Water'].MW)
    )
    fresh_solvent.imol['Methanol'] = meoh_flow_mol - recycle_solvent.imol['Methanol']
    fresh_solvent.imol['Water']    = water_flow_mol - recycle_solvent.imol['Water']
    meoh_h2o_mix.outs[0].phases = ('s', 'l', 'g')  # needed by downstream reactors

meoh_pump = bst.units.Pump('PUMP101', ins=meoh_h2o_mix-0, P=rcf_conditions['P'])

meoh_heater = bst.units.HXutility('HX102', ins=meoh_pump-0, T=rcf_conditions['T'], rigorous=True)

@meoh_heater.add_specification(run=True)
def set_meoh_heater_phases():
    meoh_heater.outs[0].phases = ('l', 'g')

# Solvolysis reactions
solvolysis_rxn = bst.Reaction(
    'Lignin -> SolubleLignin', reactant='Lignin',
    X=solvolysis_parameters['Delignification'],
    basis='wt', correct_atomic_balance=False
)
methanol_decomposition_rxn = bst.ParallelReaction([
    bst.Reaction('Methanol,l -> Methane,g', reactant='Methanol', phases='lg',
                    X=solvolysis_parameters['MeOH_CH4'], basis='wt', correct_atomic_balance=False),
    bst.Reaction('Methanol,l -> CO,g', reactant='Methanol', phases='lg',
                    X=solvolysis_parameters['MeOH_CO'], basis='wt', correct_atomic_balance=False),
])

solvolysis_reactor = SolvolysisReactor(
    'RCF103_S',
    ins=(ins, meoh_heater-0),
    outs=('Carbohydrate_Pulp', 'Solvolysis_Liquor'),
    T=rcf_conditions['T'],
    P=rcf_conditions['P'],
    tau=rcf_conditions['tau_s'],               # 3 hr time on stream per batch
    tau_0=1,                                   # 1 hr cleaning/turnaround
    tau_residence=rcf_conditions['tau_s_res'], # 20 min hydraulic residence time
    void_frac=0.5,
    superficial_velocity=0.01,
    poplar_density=poplar_density,             # 485 kg/m³ bulk density
    free_frac=0.9,                       # 10% free headspace
    V_max_limit=V_max_limit,                   # hard upper bound on vessel volume
    reaction_1=solvolysis_rxn,
    reaction_2=methanol_decomposition_rxn,
)

# H2 mixer: adjusts fresh H2 to make up for recycle shortfall
h2_mixer = bst.units.Mixer('MIX104', ins=(hydrogen_in, hydrogen_recycle))

@h2_mixer.add_specification(run=True)
def h2_flow():
    fresh_h2 = h2_mixer.ins[0]
    recycle_h2 = h2_mixer.ins[1]
    fresh_h2.imass['Hydrogen'] = (h2_biomass_ratio * (2e6 / 24)) - recycle_h2.imass['Hydrogen']
    h2_mixer.outs[0].phase = 'g'

h2_pre_heat = bst.units.HXutility('HX105', ins=h2_mixer-0, T=rcf_conditions['T'], rigorous=True)

@h2_pre_heat.add_specification(run=True)
def set_h2_preheat_phase():
    h2_pre_heat.outs[0].phase = 'g'

# Hydrogenolysis reactions
hydrogenolysis = bst.ParallelReaction([
    bst.Reaction('SolubleLignin,l -> Propylguaiacol,l', reactant='SolubleLignin', phases='lg',
                    X=rcf_oil_yield['Monomers'] * 0.5, basis='wt', correct_atomic_balance=False),
    bst.Reaction('SolubleLignin,l -> Propylsyringol,l', reactant='SolubleLignin', phases='lg',
                    X=rcf_oil_yield['Monomers'] * 0.5, basis='wt', correct_atomic_balance=False),
    bst.Reaction('SolubleLignin,l -> Syringaresinol,l', reactant='SolubleLignin', phases='lg',
                    X=rcf_oil_yield['Dimers'] * 0.5, basis='wt', correct_atomic_balance=False),
    bst.Reaction('SolubleLignin,l -> G_Dimer,l', reactant='SolubleLignin', phases='lg',
                    X=rcf_oil_yield['Dimers'] * 0.5, basis='wt', correct_atomic_balance=False),
    bst.Reaction('SolubleLignin,l -> S_Oligomer,l', reactant='SolubleLignin', phases='lg',
                    X=rcf_oil_yield['Oligomers'] * 0.5, basis='wt', correct_atomic_balance=False),
    bst.Reaction('SolubleLignin,l -> G_Oligomer,l', reactant='SolubleLignin', phases='lg',
                    X=rcf_oil_yield['Oligomers'] * 0.5, basis='wt', correct_atomic_balance=False),
])

hydrogenolysis_reactor = HydrogenolysisReactor(
    'RCF106_H',
    ins=(solvolysis_reactor.outs[1], h2_pre_heat-0),
    P=rcf_conditions['P'],
    T=rcf_conditions['T'],
    superficial_velocity=0.003,
    reaction=hydrogenolysis,
)


R102 = bst.units.Flash('FLASH107', ins=hydrogenolysis_reactor-0, T=320, P=5e5)

pre_psa_pump = bst.units.IsentropicCompressor('PUMP108', ins=R102-0, P=5e5, vle=True)
pre_psa_flash = bst.units.Flash('FLASH109', ins=pre_psa_pump-0, T=260, P=5e5)

pre_psa_heater = bst.units.HXutility('HX110', ins=pre_psa_flash-0, T=303, rigorous=True)

@pre_psa_heater.add_specification(run=True)
def set_psa_inlet_phase():
    pre_psa_heater.outs[0].phase = 'g'

psa_system = PSA('PSA111', ins=pre_psa_heater.outs[0], outs=('', 'Purge_Light_Gases'))

h2_pump = bst.units.IsentropicCompressor('PUMP112', ins=psa_system-0, outs=hydrogen_recycle,
                                            P=3e6, vle=True)

@h2_pump.add_specification(run=True)
def set_h2_pump_phase():
    h2_pump.outs[0].phase = 'g'

crude_distillation = bst.units.BinaryDistillation(
    'DIST113', ins=R102-1,
    LHK=('Methanol', 'Water'),
    Lr=0.9995, Hr=1 - 0.967, P=101325,
    vessel_material='Stainless steel 316',
    k=2, partial_condenser=True,
)

meoh_purifier_col = bst.units.BinaryDistillation(
    'DIST114', ins=crude_distillation-0,
    outs=('', 'To_WW_Treatment'),
    LHK=('Methanol', 'Water'),
    y_top=0.9, x_bot=0.001, P=101325, k=2,
)

meoh_mixer = bst.units.Mixer('MIX116', ins=(meoh_purifier_col-0, pre_psa_flash-1), rigorous=True)

cooler_2 = bst.units.HXutility('HX117', ins=meoh_mixer.outs[0], outs=meoh_recycle,
                                V=0, rigorous=True)

water_remover = bst.units.Flash('FLASH118', ins=crude_distillation-1,
                                outs=('To_WW_Treatment_2', 'RCF_Oil'), T=400, P=101325)

wastewater_mixer = bst.Mixer(
    ins=(meoh_purifier_col.outs[1], water_remover.outs[0]), outs='RCF_WW'
)

catalyst = bst.Stream(
    'RCF_Catalyst',
    NiC=RCF_catalyst['loading'] * (feed_parameters['flow'] * 1e3) * RCF_catalyst['loading'],
    units='kg/yr', price=prices['NiC_catalyst'],
)
catalyst_stream = CatalystMixer(ins=catalyst)

# ── Assemble system ───────────────────────────────────────────────────────
rcf_system =  bst.System(
    'RCF_System',
    path=(
        meoh_h2o_mix, meoh_pump, meoh_heater, solvolysis_reactor,
        h2_mixer, h2_pre_heat, hydrogenolysis_reactor,
        R102, pre_psa_pump, pre_psa_flash, pre_psa_heater,
        psa_system, h2_pump, crude_distillation, meoh_purifier_col,
        meoh_mixer, cooler_2, water_remover, wastewater_mixer,
        catalyst_stream,
    ),
    recycle=(meoh_recycle, hydrogen_recycle),
)


In [4]:
rcf_system.simulate()

In [5]:
solvent_to_crude_ratio = 1.1            # [v/kg] volume to mass ratio (10 mL/g of biomass) from 10.1039/D2RE00275B
etoac_h2o_ratio = 1                      # [v/v] mass ratio of ethyl acetate to water from 10.1039/D2RE00275B

crude_rcf = F.RCF_Oil


etoac_rho = chems['EthylAcetate'].rho(phase = 'l', T = 298.15, P = 101325)
water_rho = chems['Water'].rho(phase = 'l', T = 298.15, P = 101325)

ethyl_acetate = bst.Stream('EthylAcetate', 
                           EthylAcetate = solvent_to_crude_ratio *crude_rcf.F_mass,
                           Water = solvent_to_crude_ratio *crude_rcf.F_mass * etoac_h2o_ratio,  
                           units = 'kg/day')
solvent_recycle = bst.Stream('solvent_recycle')

with bst.System('rcf_oil_purification_sys') as rcf_oil_purification_sys:

    IDz = (
        'Water',
        'Propylguaiacol',
        'Propylsyringol',
        'Syringaresinol',
        'G_Dimer',
        'S_Oligomer',
        'G_Oligomer',
    )

    Kz = np.array([     # Array just has placeholder values now, can change depending on more granular data, but my assumption is that the separation is really good
        1.0,            # Water prefers aqueous (raffinate) phase
        200.0,          # Propylguaiacol into EtOAc (extract) > 1  
        200.0,          # Propylsyringol
        500.0,          # Syringaresinol
        109.0,          # G_Dimer
        200.0,          # S_Oligomer
        200.0,          # G_Oligomer
    ], dtype=float)

    partition_dataz = {
        'K': Kz,
        'IDs': IDz,
        'raffinate_chemicals': ['Water'],       
        'extract_chemicals':  ['EthylAcetate'], 
    }

    mixer = bst.Mixer(ins = (ethyl_acetate, solvent_recycle), rigorous = False)
    @mixer.add_specification(run = True)
    def adjust_fresh_solvent_flow():
        fresh_solvent = mixer.ins[0]
        recycled_solvent = mixer.ins[1]

        solvent_req =   solvent_to_crude_ratio *crude_rcf.F_mass
        fresh_solvent.imass['EthylAcetate'] = solvent_req - recycled_solvent.imass['EthylAcetate']

        water_req =  solvent_to_crude_ratio *crude_rcf.F_mass * etoac_h2o_ratio
        fresh_solvent.imass['Water'] = water_req - recycled_solvent.imass['Water']

    lle_column = bst.MultiStageMixerSettlers(
        'LLE_column',
        ins=(F.RCF_Oil, mixer-0),
        outs=('', 'WastePulp'),
        feed_stages=(0, -1),
        N_stages=3,                     # 3 times washed with EtOAC as proposed in ACS Sustainable Chem. Eng. 2024, 12, 12919−12926
    partition_data = partition_dataz,


        use_cache=True,
    )

    oil_flash = bst.units.Flash(        #  I use a flash rather than a distillation column because the components of RCF oil aren't defined in terms of vle data 
        ins=lle_column-0,
        outs = ('', 'Purified_RCF_Oil'),
        T = 400, 
        P = 101325
    )

    HX201 = bst.HXutility(
        ins = oil_flash-0,
        outs = '',
        V = 0,
        rigorous = True
    )

    decanter = bst.LiquidsSplitCentrifuge(ins = HX201-0, outs = (solvent_recycle, 'WW_10'), split = {'EthylAcetate':0.95} )   # Decanter that perfectly separates the Ethyl Acetate from the Water



    #water_purifier = bst.units.Flash(        #  I use a flash rather than a distillation column because the components of RCF oil aren't defined in terms of vle data 
    #    ins=lle_column-1,
    #    outs = ('WW_2', 'waste_pulp'),
    #    T = 400, 
    #    P = 101325
    #)

rcf_oil_purification_sys.simulate()

In [6]:
solvent_to_oil      = hexane_purification['solvent_to_oil_ratio']
water_hexane_ratio  = hexane_purification['water_hexane_ratio']
N_stages            = hexane_purification['N_stages']
recycle_split       = hexane_purification['hexane_recycle_split']
oil_flash_T         = hexane_purification['oil_flash_T']
oil_flash_P         = hexane_purification['oil_flash_P']
raffinate_flash_T   = hexane_purification['raffinate_flash_T']
raffinate_flash_P   = hexane_purification['raffinate_flash_P']

purified_rcf = F.Purified_RCF_Oil


hexane_rho = chems['Hexane'].rho(phase='l', T=298.15, P=101325)
water_rho  = chems['Water'].rho(phase='l', T=298.15, P=101325)


with bst.System('monomer_purification_sys') as monomer_purification_sys:

    partition_data = {
        'K':                   np.array(hexane_partition_K, dtype=float),
        'IDs':                 hexane_partition_IDs,
        'raffinate_chemicals': ['Water'],
        'extract_chemicals':   ['Hexane'],
    }

    # ── Streams ───────────────────────────────────────────────────────────────
    hexane_recycle = bst.Stream('hexane_recycle')
    hexane_in = bst.Stream(
        'Hexane_In',
        Hexane=solvent_to_oil * purified_rcf.F_mass,
        Water=solvent_to_oil * purified_rcf.F_mass * (water_rho / hexane_rho) * water_hexane_ratio,
        units='kg/hr',
        price=prices['Hexane'],
    )

    # ── Unit operations ───────────────────────────────────────────────────────

    # Fresh hexane makeup + recycle; spec sets makeup to cover the deficit each iteration
    hexane_mixer = bst.Mixer('MIX300', ins=(hexane_in, hexane_recycle), rigorous=False)

    @hexane_mixer.add_specification(run=True)
    def adjust_fresh_solvent_flow():
        fresh   = hexane_mixer.ins[0]
        recycle = hexane_mixer.ins[1]
        fresh.imass['Hexane'] = (
            solvent_to_oil * purified_rcf.F_mass - recycle.imass['Hexane']
        )
        fresh.imass['Water'] = (
            solvent_to_oil * purified_rcf.F_mass * (water_rho / hexane_rho) * water_hexane_ratio
            - recycle.imass['Water']
        )

    # LLE: purified oil contacts hexane/water countercurrently; monomers and dimers
    # partition into hexane extract; oligomers (unlisted) remain in aqueous raffinate
    lle_column = bst.MultiStageMixerSettlers(
        'LLE300',
        ins=(purified_rcf, hexane_mixer-0),
        outs=('', ''),
        feed_stages=(0, -1),
        N_stages=N_stages,
        partition_data=partition_data,
        use_cache=True,
    )

    # Flash hexane overhead; monomers/dimers exit as bottoms
    monomer_flash = bst.units.Flash(
        'FLASH301',
        ins=lle_column-0,
        outs=('', 'RCF_Monomers'),
        T=oil_flash_T,
        P=oil_flash_P,
    )

    # Condense hexane vapor for decanting
    solvent_cooler = bst.units.HXutility(
        'HX302',
        ins=monomer_flash-0,
        V=0,
        rigorous=True,
    )

    # Split hexane from water; hexane-rich phase recycled, water bleed to wastewater
    solvent_decanter = bst.LiquidsSplitCentrifuge(
        'CENT303',
        ins=solvent_cooler-0,
        outs=(hexane_recycle, 'WW_11'),
        split={'Hexane': recycle_split},
    )

    # Flash raffinate to separate oligomers from wastewater; overhead water to WWT
    raffinate_flash = bst.units.Flash(
        'FLASH304',
        ins=lle_column-1,
        outs=('WW_21', 'RCF_Oligomers'),
        T=raffinate_flash_T,
        P=raffinate_flash_P,
    )

monomer_purification_sys.simulate()


In [7]:
lle_column

MultiStageMixerSettlers: LLE300
ins...
[0] Purified_RCF_Oil  from  Flash-F1
    phase: 'l', T: 400 K, P: 101325 Pa
    flow (kmol/hr): Propylguaiacol  25
                    Propylsyringol  21.2
                    Syringaresinol  4.97
                    G_Dimer         5.73
                    S_Oligomer      3.39
                    G_Oligomer      3.84
[1] s24  from  Mixer-MIX300
    phase: 'l', T: 309.49 K, P: 101325 Pa
    flow (kmol/hr): Water   7.02e+03
                    Hexane  965
outs...
[0] s25  to  Flash-FLASH301
    phase: 'l', T: 334.55 K, P: 101325 Pa
    flow (kmol/hr): Water           0.267
                    Hexane          965
                    Propylguaiacol  25
                    Propylsyringol  21.2
                    Syringaresinol  4.96
                    G_Dimer         5.73
[1] s26  to  Flash-FLASH304
    phase: 'l', T: 309.55 K, P: 101325 Pa
    flow (kmol/hr): Water           7.02e+03
                    Propylguaiacol  0.00816
                    P

In [8]:
BT, WWT, gas_mixer = create_rcf_utilities_system()

In [9]:
monomer_purification_sys

System: monomer_purification_sys
Highest convergence error among components in recycle
stream CENT303-0 after 2 loops:
- flow rate   1.14e-13 kmol/hr (1.1e-11%)
- temperature 3.51e-09 K (1e-09%)
ins...
[0] Hexane_In  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water   7.02e+03
                    Hexane  48.2
[1] Purified_RCF_Oil  from  Flash-F1
    phase: 'l', T: 400 K, P: 101325 Pa
    flow (kmol/hr): Propylguaiacol  25
                    Propylsyringol  21.2
                    Syringaresinol  4.97
                    G_Dimer         5.73
                    S_Oligomer      3.39
                    G_Oligomer      3.84
outs...
[0] WW_11  
    phase: 'l', T: 341.78 K, P: 101325 Pa
    flow (kmol/hr): Water   0.267
                    Hexane  48.2
[1] RCF_Monomers  
    phase: 'l', T: 400 K, P: 101325 Pa
    flow (kmol/hr): Propylguaiacol  25
                    Propylsyringol  21.2
                    Syringaresinol  4.96
                    G_Dimer         5.73
[

In [10]:
rcf_oil_purification_sys

System: rcf_oil_purification_sys
Highest convergence error among components in recycle
stream M3-0 after 4 loops:
- flow rate   5.68e-14 kmol/hr (5.7e-12%)
- temperature 1.71e-13 K (5.5e-14%)
ins...
[0] EthylAcetate  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water         1.52e+03
                    EthylAcetate  15.5
[1] RCF_Oil  from  Flash-FLASH118
    phase: 'l', T: 400 K, P: 101325 Pa
    flow (kmol/hr): Extract         7.4
                    Glucan          23.8
                    Xylan           5.92
                    Arabinan        0.757
                    Mannan          9.51
                    Galactan        3.6
                    Propylguaiacol  25
                    Propylsyringol  21.2
                    Syringaresinol  4.97
                    G_Dimer         5.73
                    S_Oligomer      3.39
                    G_Oligomer      3.84
outs...
[0] WW_10  
    phase: 'l', T: 344.74 K, P: 101325 Pa
    flow (kmol/hr): Water         

In [11]:
solvolysis_reactor

SolvolysisReactor: RCF103_S
ins...
[0] Poplar_In  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water     925
                    Sucrose   0.243
                    Extract   7.4
                    Acetate   48.6
                    Ash       1e+03
                    Lignin    156
                    Glucan    238
                    Xylan     84.5
                    Arabinan  1.26
                    Mannan    19
                    Galactan  7.2
[1] s3  from  HXutility-HX102
    phases: ('g', 'l'), T: 473.15 K, P: 6e+06 Pa
    flow (kmol/hr): (g) Water     0.0714
                        Acetate   4.73e-08
                        Methanol  0.468
                        Hydrogen  7.85e-09
                        Methane   0.35
                    (l) Water     7.07e+03
                        Acetate   0.0431
                        Methanol  2.39e+04
                        Hydrogen  2.87e-17
                        Methane   0.132
outs...
[0] Carbohydrate_Pulp  


In [12]:
rcf_combined_system = bst.System(
    'Combined_RCF_System',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys, WWT),
    facilities=[gas_mixer, BT],
)
rcf_combined_system.simulate()

In [13]:
rcf_combined_system.diagram()


In [14]:
rcf_combined_system

System: Combined_RCF_System
Highest convergence error among components in recycle
streams {HX117-0, PUMP112-0} after 1 loops:
- flow rate   2.05e-07 kmol/hr (0.012%)
- temperature 1.18e-03 K (0.00037%)
ins...
[0] RCF_Catalyst  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): NiC  2.28
[1] EthylAcetate  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water         1.52e+03
                    EthylAcetate  15.5
[2] -  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow: 0
[3] -  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow: 0
[4] air_lagoon  
    phase: 'g', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): N2  8.01
                    O2  1.98
[5] caustic  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water  26.5
                    NaOH   11.9
[6] Poplar_In  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water     925
                    Sucrose   0.243
                    Extract   7.4
                    Acetate   48.6

In [15]:
F.WastePulp

Stream: WastePulp from <MultiStageMixerSettlers: LLE_column>
phase: 'l', T: 322.72 K, P: 101325 Pa
flow (kmol/hr): Water           1.43e+03
                Extract         7.4
                Glucan          23.8
                Xylan           5.92
                Arabinan        0.757
                Mannan          9.51
                Galactan        3.6
                Propylguaiacol  1.4e-05
                Propylsyringol  1.18e-05
                Syringaresinol  1.8e-07
                G_Dimer         1.93e-05
                S_Oligomer      1.89e-06
                G_Oligomer      2.15e-06


In [16]:
F.BT

BoilerTurbogenerator: BT
ins...
[0] sludge  from  SludgeCentrifuge-S603
    phase: 'l', T: 308.12 K, P: 101325 Pa
    flow (kmol/hr): Water      21.1
                    Acetate    1.19
                    NaOH       0.182
                    WWTsludge  0.944
                    Methanol   0.00744
[1] s43  from  Mixer-MIX_BT_gas
    phase: 'g', T: 490.87 K, P: 101325 Pa
    flow (kmol/hr): Water     0.897
                    CH4       7.57
                    CO2       7.28
                    CO        99.7
                    Acetate   0.0182
                    Methanol  5.82
                    Hydrogen  258
                    Methane   61.2
[2] -  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water  2.44e+03
[3] -  
    phase: 'g', T: 288.71 K, P: 101560 Pa
    flow (kmol/hr): CH4  4.73e+03
[4] -  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water  0.0159
                    Lime   0.00396
[5] -  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    fl

In [17]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
solvent_to_lignin_oil = 5               #  [kg/kg] massratio of solvent to lignin oil from https://doi.org/10.1126/science.aau1567
water_hexane_ratio = 1                  # [v/v] volume ratio of hexane to water

pure_rcf = F.Purified_RCF_Oil

hexane_rho = chems['Hexane'].rho(phase = 'l', T = 298.15, P = 101325)
water_rho = chems['Water'].rho(phase = 'l', T = 298.15, P = 101325)

hexane = bst.Stream('Hexane', 
                           Hexane = solvent_to_lignin_oil * pure_rcf.F_mass, 
                           Water = (solvent_to_lignin_oil * pure_rcf.F_mass) * (water_rho/hexane_rho) * water_hexane_ratio,
                           units = 'kg/hr')
hexane_recycle = bst.Stream('hexane_recycle')

with bst.System('monomer_purification_sys') as monomer_purification_sys:

    IDzz = (
        'Water',
        'Propylguaiacol',
        'Propylsyringol',
        'Syringaresinol',
        'G_Dimer',

        
    )

    Kzz = np.array([     # Array just has placeholder values now, can change depending on more granular data, but my assumption is that the separation is really effective
        10.0,            # Water prefers aqueous (raffinate) phase
        2.0,          # Propylguaiacol into hexane 
        2.0,          # Propylsyringol
        2.0,          # Syringaresinol
        2.0,          # G-Dimer

    ], dtype=float)

    partition_datazz = {
        'K': Kzz,
        'IDs': IDzz,
        'raffinate_chemicals': ['Water'],       
        'extract_chemicals':  ['Hexane'], 
    }

    hexane_mixer = bst.Mixer(ins = (hexane, hexane_recycle), rigorous = False)
    @hexane_mixer.add_specification(run = True)
    def adjust_fresh_solvent_flow():
        fresh_solvent = hexane_mixer.ins[0]
        recycled_solvent = hexane_mixer.ins[1]

        solvent_req =  solvent_to_lignin_oil * pure_rcf.F_mass
        fresh_solvent.imass['Hexane'] = solvent_req - recycled_solvent.imass['Hexane']

        water_req =  (solvent_to_lignin_oil * pure_rcf.F_mass) * (water_rho/hexane_rho) * water_hexane_ratio
        fresh_solvent.imass['Water'] = water_req - recycled_solvent.imass['Water']

    #mixer.simulate()

    lle_column_2 = bst.MultiStageMixerSettlers(
        'LLE_column_2',
        ins=(F.Purified_RCF_Oil, hexane_mixer-0),
        outs=('extract', 'raffinate'),
        feed_stages=(0, -1),
        N_stages=3,                     # 3 times washed with EtOAC as proposed in ACS Sustainable Chem. Eng. 2024, 12, 12919−12926
    partition_data = partition_datazz,


        use_cache=True,
    )
    #lle_column.simulate()

    oil_flash_2 = bst.units.Flash(        #  I use a flash rather than a distillation column because the components of RCF oil aren't defined in terms of vle data 
        ins=lle_column_2-0,
        outs = ('', 'RCF_Monomers'),
        T = 400, 
        P = 101325
    )
    #oil_flash.simulate()

    HX202 = bst.HXutility(
        ins = oil_flash_2-0,
        outs = '',
        V = 0,
        rigorous = True
    )
    #HX201.simulate()
    decanter_2 = bst.LiquidsSplitCentrifuge(ins = HX202-0, outs = (hexane_recycle, 'WW_11'), split = {'Hexane':0.95} )   # Decanter that perfectly separates the Ethyl Acetate from the Water
    #decanter.simulate()


    water_purifier_2 = bst.units.Flash(        #  I use a flash rather than a distillation column because the components of RCF oil aren't defined in terms of vle data
        ins=lle_column_2-1,
        outs = ('WW_21', 'oligomers'),
        T = 400,
        P = 101325
    )

monomer_purification_sys.simulate()
#rcf_oil_purification_sys.diagram(kind='cluster', format='png')
#rcf_oil_purification_sys.diagram()